# AI Youth & Employment Scheme Navigator - Eligibility Engine

This notebook implements a deterministic matching and scoring engine that evaluates a user's eligibility profile against a database of government schemes (`schemes.json`).

### Eligibility scoring weights:
* **Age Match**: 20 points
* **Occupation Match**: 20 points
* **Education Match**: 20 points
* **Income Match**: 20 points
* **State Match**: 20 points
* **Total Possible Score**: 100 points

*Note: Gender acts as a hard requirement filter (schemes with strict gender eligibility restrictions are filtered out).* 

---

## 1. Import Libraries

First, we import the necessary Python libraries: `json` for serialization and `pandas` for processing the data in a tabular format.

In [1]:
import json
import pandas as pd

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load the Dataset

We load the validated schemes database (`schemes.json`) into a Pandas DataFrame.

In [2]:
# Load schemes.json into a DataFrame
with open("schemes.json", "r", encoding="utf-8") as f:
    schemes_data = json.load(f)

df_schemes = pd.DataFrame(schemes_data)
print(f"Loaded {len(df_schemes)} schemes successfully!")
df_schemes.head(2)

Loaded 20 schemes successfully!


,scheme_id,scheme_name,category,ministry,description,min_age,max_age,gender,occupation,education,income_limit,state,benefits,required_documents,application_link,official_scheme_url
0,SCH-001,Pradhan Mantri Kaushal Vikas Yojana (PMKVY),Skill Development,Ministry of Skill Development and Entrepreneur...,Skill certification scheme aiming to enable In...,15,45.0,All,Unemployed / Students,8th standard and above,None,All India,[Free skill training courses aligned with NSQF...,"[Aadhaar Card, Educational Qualification Certi...",https://www.pmkvyofficial.org/apply-now,https://www.pmkvyofficial.org/
1,SCH-002,National Apprenticeship Promotion Scheme (NAPS),Apprenticeships,Ministry of Skill Development and Entrepreneur...,Scheme to promote apprenticeship training and ...,14,35.0,All,Students / Job Seekers,5th class pass to ITI/Diploma/Graduates,None,All India,"[On-the-job training in recognized industries,...","[Aadhaar Card, Educational Qualification Certi...",https://www.apprenticeshipindia.gov.in/candida...,https://www.apprenticeshipindia.gov.in/


## 3. Define the User Profile

We create the sample user profile representing a 22-year-old male student from Delhi who is a graduate with an annual income of Rs 3,00,000.

In [3]:
user_profile = {
    "age": 22,
    "gender": "male",
    "occupation": "student",
    "education": "graduate",
    "annual_income": 300000,
    "state": "Delhi"
}

print("Sample User Profile:")
print(json.dumps(user_profile, indent=4))

Sample User Profile:
{
    "age": 22,
    "gender": "male",
    "occupation": "student",
    "education": "graduate",
    "annual_income": 300000,
    "state": "Delhi"
}


## 4. Implement Eligibility Checkers

We build the modular deterministic functions for matching each field of the user profile against the schemes' criteria.

In [4]:
def check_age_eligible(user_age: int, min_age, max_age) -> bool:
    """True if user age falls within scheme age boundaries (inclusive)."""
    eligible_min = True if min_age is None or pd.isna(min_age) else user_age >= int(min_age)
    eligible_max = True if max_age is None or pd.isna(max_age) else user_age <= int(max_age)
    return eligible_min and eligible_max

def check_gender_eligible(user_gender: str, scheme_gender) -> bool:
    """True if scheme has no strict gender limitation or matches the user's gender."""
    user_gender = user_gender.lower().strip()
    if isinstance(scheme_gender, list):
        scheme_genders = [g.lower().strip() for g in scheme_gender]
    else: 
        scheme_genders = [g.lower().strip() for g in str(scheme_gender).replace(",", "/").split("/")]
        
    if "all" in scheme_genders or "any" in scheme_genders:
        return True
    return user_gender in scheme_genders

def check_occupation_eligible(user_occ: str, scheme_occ) -> bool:
    """True if scheme covers user's occupation, or is open to all/any occupations."""
    user_occ = user_occ.lower().strip()
    if isinstance(scheme_occ, list):
        scheme_occs = [o.lower().strip() for o in scheme_occ]
    else:
        scheme_occs = [o.lower().strip() for o in str(scheme_occ).replace(",", "/").split("/")]
        
    if any(any(x in occ for x in ["all", "any", "not specified", "not required"]) for occ in scheme_occs):
        return True
    return any(user_occ in occ or occ in user_occ for occ in scheme_occs)

def check_education_eligible(user_edu: str, scheme_edu) -> bool:
    """True if user's education meets or exceeds the minimum required education of the scheme."""
    user_edu = user_edu.lower().strip()
    scheme_edu_str = str(scheme_edu).lower().strip()
    
    if any(x in scheme_edu_str for x in ["not required", "not specified", "literate", "all"]):
        return True
        
    ranks = {
        "literate": 0,
        "5th": 1, "five": 1, "class 5": 1,
        "8th": 2, "eight": 2, "class 8": 2,
        "10th": 3, "ten": 3, "matric": 3, "secondary": 3, "class 10": 3, " 10": 3,
        "12th": 4, "twelve": 4, "higher secondary": 4, "class 12": 4, " 12": 4,
        "iti": 5, "diploma": 5,
        "graduate": 6, "degree": 6, "undergraduate": 6, "college": 6, "bachelor": 6,
        "postgraduate": 7, "master": 7, "post graduate": 7
    }
    
    user_rank = 0
    for key, rank in ranks.items():
        if key in user_edu:
            user_rank = max(user_rank, rank)
            
    scheme_rank_required = 0
    has_requirement = False
    for key, rank in ranks.items():
        if key in scheme_edu_str:
            has_requirement = True
            scheme_rank_required = max(scheme_rank_required, rank)
            
    if has_requirement:
        return user_rank >= scheme_rank_required
        
    return user_edu in scheme_edu_str

def parse_income_limit(limit):
    """Helper to convert string/numeric income limits into float value."""
    if limit is None or pd.isna(limit):
        return None
    try:
        return float(limit)
    except ValueError:
        pass
        
    limit_str = str(limit).lower().replace(",", "").replace(" ", "")
    multiplier = 1
    if "lakh" in limit_str:
        multiplier = 100000
        limit_str = limit_str.replace("lakh", "").replace("s", "")
        
    nums = "".join([c for c in limit_str if c.isdigit() or c == '.'])
    if nums:
        try:
            return float(nums) * multiplier
        except ValueError:
            pass
    return None

def check_income_eligible(user_income: float, income_limit) -> bool:
    """True if user's annual income is within the scheme's maximum income limit."""
    limit_val = parse_income_limit(income_limit)
    if limit_val is None:
        return True
    return user_income <= limit_val

def check_state_eligible(user_state: str, scheme_state: str) -> bool:
    """True if scheme state matches user's state or applies to 'All India'."""
    user_state = user_state.lower().strip()
    scheme_state_str = str(scheme_state).lower().strip()
    if "all india" in scheme_state_str or "any" in scheme_state_str:
        return True
    return user_state in scheme_state_str

print("All eligibility checker functions defined successfully!")

All eligibility checker functions defined successfully!


## 5. Score and Match Schemes

We loop through the database, evaluate each checker function, apply the gender hard-filter, and calculate final match scores and reasons.

In [5]:
results = []

for _, row in df_schemes.iterrows():
    # Evaluate all criteria
    age_ok = check_age_eligible(user_profile["age"], row["min_age"], row["max_age"])
    gender_ok = check_gender_eligible(user_profile["gender"], row["gender"])
    occ_ok = check_occupation_eligible(user_profile["occupation"], row["occupation"])
    edu_ok = check_education_eligible(user_profile["education"], row["education"])
    inc_ok = check_income_eligible(user_profile["annual_income"], row["income_limit"])
    state_ok = check_state_eligible(user_profile["state"], row["state"])
    
    # Gender is a hard requirement for eligibility
    if not gender_ok:
        continue
        
    score = 0
    reasons = []
    
    # Calculate score & compile reasoning logs
    if age_ok:
        score += 20
        reasons.append("Age eligible")
    if occ_ok:
        score += 20
        reasons.append("Occupation eligible")
    if edu_ok:
        score += 20
        reasons.append("Education eligible")
    if inc_ok:
        score += 20
        reasons.append("Income eligible")
    if state_ok:
        score += 20
        reasons.append("State eligible")
        
    if score > 0:
        results.append({
            "Scheme Name": row["scheme_name"],
            "Category": row["category"],
            "Match Score": score,
            "Eligibility Reasons": reasons,
            "Benefits": row["benefits"] if isinstance(row["benefits"], str) else ", ".join(row["benefits"])
        })

print(f"Eligibility calculation complete! Found {len(results)} matching schemes.")

Eligibility calculation complete! Found 17 matching schemes.


## 6. Sort, Format, and Display Ranked Results

We convert the results to a DataFrame, sort descending by score, format reasons, and display the final ranked output table.

In [6]:
# Load into DataFrame and sort by Match Score descending
df_results = pd.DataFrame(results).sort_values(by="Match Score", ascending=False)

# Format eligibility reasons list to a clean string
df_results["Eligibility Reasons"] = df_results["Eligibility Reasons"].apply(lambda r: ", ".join(r))

# Display results table
df_results

,Scheme Name,Category,Match Score,Eligibility Reasons,Benefits
0,Pradhan Mantri Kaushal Vikas Yojana (PMKVY),Skill Development,100,"Age eligible, Occupation eligible, Education e...","Free skill training courses aligned with NSQF,..."
1,National Apprenticeship Promotion Scheme (NAPS),Apprenticeships,100,"Age eligible, Occupation eligible, Education e...","On-the-job training in recognized industries, ..."
4,Pradhan Mantri YUVA Yojana (PM-YUVA),Skill Development,100,"Age eligible, Occupation eligible, Education e...","Structured entrepreneurship education courses,..."
13,STRIVE (Skills Strengthening for Industrial Va...,Apprenticeships,100,"Age eligible, Occupation eligible, Education e...","Access to modern workshops, updated curriculum..."
11,Central Sector Scheme of Scholarship for Colle...,Scholarships,100,"Age eligible, Occupation eligible, Education e...","Scholarship of Rs 12,000 per annum for the fir..."
9,National Career Service (NCS),Employment Assistance,80,"Age eligible, Occupation eligible, Income elig...","Free job placement and matching services, Acce..."
10,Mahatma Gandhi National Rural Employment Guara...,Employment Assistance,80,"Age eligible, Education eligible, Income eligi...",100 days of guaranteed unskilled manual labour...
2,Prime Minister's Internship Scheme (PMIS),Internships,80,"Age eligible, Education eligible, Income eligi...","Monthly stipend of Rs 5,000 for 12 months, One..."
3,Deen Dayal Upadhyaya Grameen Kaushalya Yojana ...,Skill Development,80,"Age eligible, Education eligible, Income eligi...","Completely free skill training programs, Free ..."
12,SANKALP (Skills Acquisition and Knowledge Awar...,Skill Development,80,"Age eligible, Education eligible, Income eligi...",Access to industry-aligned certified vocationa...
